# Phase 3 — Adding clinical workflow features

Phases 1 and 2 used clinical *values* (vital signs, lab results, drug counts per hour, etc.) as features. The model can learn relationships between these values and the sepsis label, but it has no view into **patterns of clinical decision-making over time**:

- How long has the patient been in the ICU at this hour?
- How many drug administrations happened in the past 6 hours?
- How many distinct lab tests have been ordered recently?
- Are invasive devices currently in use?

These *workflow* signals are clinically meaningful — a clinician ordering many labs and drugs in a short window is concerned about something — but they require integrating events across multiple rows of the patient's history, which gradient-boosted trees can't extract from per-(patient, hour) features alone.

**This notebook builds on Phase 2** (Khang's features + CatBoost) and adds the following workflow features:
1. `hour_in_stay` — hours since the patient's first record (causal proxy for hours-since-admission)
2. `log_hour_in_stay`, `is_first_24h` — transformations of #1
3. `drug_admin_count_6h`, `drug_admin_count_24h` — drug-administration intensity in past 6 / 24 hours
4. `proc_count_24h` — number of procedure events in past 24 hours
5. `n_active_devices_24h` — count of distinct devices recorded in past 24 hours
6. `has_et_tube_24h`, `has_cvl_24h` — binary flags for invasive devices in past 24 hours

All features are causal — they only use information from time ≤ t.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier

# Reuse Khang's pipeline
from missing_values import prepare_for_xgboost
from XGBoost.train_XGBoost import load_training_data, split_data_person_aware, score_model

ARTIFACTS = Path('artifacts')
ROOT = Path('phems-hackathon-early-sepsis-prediction')
TRAIN_DIR = ROOT / 'training_data'
TEST_DIR  = ROOT / 'testing_data'
RANDOM_SEED = 42
print('Setup OK')

Setup OK


## 1. Compute workflow features

Each function below takes a master `(person_id, measurement_datetime)` index and returns those same rows with a workflow-feature column attached. We apply the same logic to the training and test feature tables.

In [2]:
def floor_to_hour(s):
    return pd.to_datetime(s, errors='coerce').dt.floor('h')

def add_hour_in_stay(df):
    """Hours since this patient's earliest record."""
    df = df.sort_values(['person_id', 'measurement_datetime']).reset_index(drop=True)
    first_t = df.groupby('person_id')['measurement_datetime'].transform('min')
    df['hour_in_stay']     = ((df['measurement_datetime'] - first_t).dt.total_seconds() / 3600).astype('float32')
    df['log_hour_in_stay'] = np.log1p(df['hour_in_stay']).astype('float32')
    df['is_first_24h']     = (df['hour_in_stay'] < 24).astype('int8')
    return df

def add_event_count_in_window(df, source_df, source_dt_col, window_hours, feat_name):
    """Count source_df events per (patient, hour) within the past `window_hours`."""
    src = source_df[['person_id', source_dt_col]].copy()
    src['t_event'] = floor_to_hour(src[source_dt_col])
    src = src.dropna(subset=['t_event'])
    hourly = (src.assign(_n=1).groupby(['person_id', 't_event'])['_n']
                .sum().reset_index()
                .rename(columns={'t_event': 'measurement_datetime'}))
    out = df.merge(hourly, on=['person_id', 'measurement_datetime'], how='left')
    out['_n'] = out['_n'].fillna(0).astype('float32')
    out = out.sort_values(['person_id', 'measurement_datetime']).reset_index(drop=True)
    out[feat_name] = (out.groupby('person_id', sort=False)['_n']
                       .rolling(window=window_hours, min_periods=1)
                       .sum().reset_index(level=0, drop=True).astype('float32'))
    return out.drop(columns='_n')

def add_distinct_devices_in_window(df, devices_df, window_hours, feat_name):
    """Count of distinct device types recorded in past `window_hours` for the patient."""
    src = devices_df[['person_id', 'device_datetime_hourly', 'device']].copy()
    src['t_event'] = floor_to_hour(src['device_datetime_hourly'])
    src = src.dropna(subset=['t_event', 'device'])
    distinct_per_hr = (src.groupby(['person_id', 't_event'])['device']
                          .nunique().reset_index()
                          .rename(columns={'t_event': 'measurement_datetime', 'device': '_n_distinct'}))
    out = df.merge(distinct_per_hr, on=['person_id', 'measurement_datetime'], how='left')
    out['_n_distinct'] = out['_n_distinct'].fillna(0).astype('float32')
    out = out.sort_values(['person_id', 'measurement_datetime']).reset_index(drop=True)
    out[feat_name] = (out.groupby('person_id', sort=False)['_n_distinct']
                       .rolling(window=window_hours, min_periods=1)
                       .sum().reset_index(level=0, drop=True).astype('float32'))
    return out.drop(columns='_n_distinct')

def add_invasive_device_flag(df, devices_df, device_name, lookback_hours, feat_name):
    """Binary flag: was this specific device recorded in the past `lookback_hours`?"""
    sub = devices_df[devices_df['device'] == device_name].copy()
    sub['t_event'] = floor_to_hour(sub['device_datetime_hourly'])
    counts = (sub.assign(_n=1).groupby(['person_id', 't_event'])['_n']
                .sum().reset_index()
                .rename(columns={'t_event': 'measurement_datetime'}))
    out = df.merge(counts, on=['person_id', 'measurement_datetime'], how='left')
    out['_n'] = out['_n'].fillna(0).astype('float32')
    out = out.sort_values(['person_id', 'measurement_datetime']).reset_index(drop=True)
    rolling = (out.groupby('person_id', sort=False)['_n']
                  .rolling(window=lookback_hours, min_periods=1)
                  .sum().reset_index(level=0, drop=True))
    out[feat_name] = (rolling > 0).astype('int8')
    return out.drop(columns='_n')

print('Workflow helpers defined.')

Workflow helpers defined.


In [3]:
def build_workflow_features(master_df, drugs_df, procs_df, devices_df):
    """Add all workflow features to a master (person_id, measurement_datetime) table."""
    out = master_df.copy()
    out['measurement_datetime'] = pd.to_datetime(out['measurement_datetime'])
    out = add_hour_in_stay(out)
    out = add_event_count_in_window(out, drugs_df,  'drug_datetime_hourly',      6,  'drug_admin_count_6h')
    out = add_event_count_in_window(out, drugs_df,  'drug_datetime_hourly',      24, 'drug_admin_count_24h')
    out = add_event_count_in_window(out, procs_df,  'procedure_datetime_hourly', 24, 'proc_count_24h')
    out = add_distinct_devices_in_window(out, devices_df, 24, 'n_active_devices_24h')
    out = add_invasive_device_flag(out, devices_df, 'Endotracheal tube',       24, 'has_et_tube_24h')
    out = add_invasive_device_flag(out, devices_df, 'Central venous catheter', 24, 'has_cvl_24h')
    return out

print('Loading raw event tables for both splits...')
drugs_tr   = pd.read_csv(TRAIN_DIR / 'drugsexposure_train.csv')
procs_tr   = pd.read_csv(TRAIN_DIR / 'proceduresoccurrences_train.csv')
devices_tr = pd.read_csv(TRAIN_DIR / 'devices_train.csv')
drugs_te   = pd.read_csv(TEST_DIR  / 'drugsexposure_test.csv')
procs_te   = pd.read_csv(TEST_DIR  / 'proceduresoccurrences_test.csv')
devices_te = pd.read_csv(TEST_DIR  / 'devices_test.csv')
print(f'  drugs    train={drugs_tr.shape}, test={drugs_te.shape}')
print(f'  procs    train={procs_tr.shape}, test={procs_te.shape}')
print(f'  devices  train={devices_tr.shape}, test={devices_te.shape}')

Loading raw event tables for both splits...


  drugs    train=(184780, 5), test=(74801, 5)
  procs    train=(771214, 4), test=(300447, 4)
  devices  train=(750878, 4), test=(320919, 4)


## 2. Build augmented training features
Reuse Khang's `load_training_data()` for the base, then merge in workflow features. The original row order must be preserved so labels and features stay aligned.

In [4]:
df, base_features, labels = load_training_data()
print(f'Base features: {base_features.shape}')

# Build workflow features keyed on the UNIQUE (person_id, measurement_datetime) pairs.
# Khang's train_features.csv has 30 rows with duplicate (person_id, hour) keys; if we
# merge a non-deduped wf table back, we'd produce 30 extra rows from the cross-product.
master_keys = (df[['person_id', 'measurement_datetime']]
               .drop_duplicates()
               .reset_index(drop=True))
master_keys['measurement_datetime'] = pd.to_datetime(master_keys['measurement_datetime'])

wf_train = build_workflow_features(master_keys, drugs_tr, procs_tr, devices_tr)
wf_cols = ['hour_in_stay', 'log_hour_in_stay', 'is_first_24h',
           'drug_admin_count_6h', 'drug_admin_count_24h',
           'proc_count_24h', 'n_active_devices_24h',
           'has_et_tube_24h', 'has_cvl_24h']

# Align workflow features to df's row order (handles duplicate keys correctly via left-join)
df_keyed = df[['person_id', 'measurement_datetime']].copy()
df_keyed['measurement_datetime'] = pd.to_datetime(df_keyed['measurement_datetime'])
wf_aligned = df_keyed.merge(
    wf_train[['person_id', 'measurement_datetime'] + wf_cols],
    on=['person_id', 'measurement_datetime'], how='left',
)[wf_cols].reset_index(drop=True)
assert len(wf_aligned) == len(base_features), f'Length mismatch after alignment: {len(wf_aligned)} vs {len(base_features)}'
wf_aligned.index = base_features.index

# Combine
augmented_features = pd.concat([base_features, wf_aligned], axis=1)
print(f'Augmented features: {augmented_features.shape} (added {len(wf_cols)} workflow features)')

Base features: (331653, 258)


Augmented features: (331653, 267) (added 9 workflow features)


## 3. Person-aware split

In [5]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data_person_aware(
    df, augmented_features, labels, test_size=0.2, val_size=0.2, random_state=RANDOM_SEED
)
print(f'Train: {X_train.shape}, positives: {int(y_train.sum())} ({y_train.mean():.4%})')
print(f'Val:   {X_val.shape}, positives: {int(y_val.sum())} ({y_val.mean():.4%})')
print(f'Test:  {X_test.shape}, positives: {int(y_test.sum())} ({y_test.mean():.4%})')

Train: (198331, 267), positives: 4736 (2.3879%)
Val:   (69344, 267), positives: 1164 (1.6786%)
Test:  (63978, 267), positives: 974 (1.5224%)


## 4. Train CatBoost on augmented features
Same hyperparameters as Phase 2 — only the feature set has changed.

In [6]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    auto_class_weights='Balanced',
    eval_metric='PRAUC',
    loss_function='Logloss',
    early_stopping_rounds=30,
    random_seed=RANDOM_SEED,
    verbose=False,
)
model.fit(
    X_train.to_numpy(), y_train,
    eval_set=(X_val.to_numpy(), y_val),
    verbose=False,
)
print(f'Best iteration (early-stopped): {model.get_best_iteration()}')

Best iteration (early-stopped): 16


In [7]:
print('=' * 60)
print('CATBOOST + WORKFLOW FEATURES — performance on each split')
print('=' * 60)
score_model(model, X_train, y_train, 'Train')
print()
score_model(model, X_val,   y_val,   'Validation')
print()
score_model(model, X_test,  y_test,  'Test')

CATBOOST + WORKFLOW FEATURES — performance on each split


Train ROC AUC:   0.9812
Train PR AUC:    0.7463
Train Accuracy:  0.8252 @ threshold=0.20
Train Recall:    0.9764 @ threshold=0.20
Train Precision: 0.1180 @ threshold=0.20



Validation ROC AUC:   0.9642
Validation PR AUC:    0.4172
Validation Accuracy:  0.8158 @ threshold=0.20
Validation Recall:    0.9605 @ threshold=0.20
Validation Precision: 0.0807 @ threshold=0.20



Test ROC AUC:   0.9671
Test PR AUC:    0.4383
Test Accuracy:  0.8245 @ threshold=0.20
Test Recall:    0.9733 @ threshold=0.20
Test Precision: 0.0780 @ threshold=0.20


## 5. Feature importance — did the workflow features actually matter?

In [8]:
imp = pd.Series(model.feature_importances_, index=augmented_features.columns).sort_values(ascending=False)
print('Top 15 features by importance:')
print(imp.head(15).round(2).to_string())
print()
print('Of these, which are the new workflow features?')
wf_in_top15 = [c for c in imp.head(15).index if c in wf_cols]
print(f'  {len(wf_in_top15)}: {wf_in_top15}')

Top 15 features by importance:
route_Intravenous                                22.45
drug_admin_count_6h                              13.18
log_hour_in_stay                                 12.18
drug_epinephrine                                  5.79
Measurement of oxygen saturation at periphery     4.98
age_in_months                                     3.55
drug_clindamycin                                  3.09
missing_drug_isavuconazole                        2.70
missing_drug_gentamicin                           2.62
drug_dopamine                                     2.16
n_active_devices_24h                              2.02
Systolic blood pressure                           1.94
device_Urinary catheter                           1.93
missing_drug_linezolid                            1.72
proc_count_24h                                    1.64

Of these, which are the new workflow features?
  4: ['drug_admin_count_6h', 'log_hour_in_stay', 'n_active_devices_24h', 'proc_count_24h']


## 6. Generate Kaggle submission

In [9]:
test_df = pd.read_csv(ARTIFACTS / 'test_features.csv')
test_df['measurement_datetime'] = pd.to_datetime(test_df['measurement_datetime'])
ids = test_df['person_id'].astype(str) + '_' + test_df['measurement_datetime'].astype(str)

X_kaggle_base = test_df.drop(columns=['person_id', 'measurement_datetime'], errors='ignore')
X_kaggle_base = prepare_for_xgboost(X_kaggle_base, label_column=None, add_indicators=True, add_summary=True)
X_kaggle_base = X_kaggle_base.reindex(columns=base_features.columns)

# Workflow features on unique test keys (same dedupe logic as train)
test_keys = (test_df[['person_id', 'measurement_datetime']]
             .drop_duplicates()
             .reset_index(drop=True))
wf_test = build_workflow_features(test_keys, drugs_te, procs_te, devices_te)

wf_test_aligned = test_df[['person_id', 'measurement_datetime']].merge(
    wf_test[['person_id', 'measurement_datetime'] + wf_cols],
    on=['person_id', 'measurement_datetime'], how='left',
)[wf_cols].reset_index(drop=True)
assert len(wf_test_aligned) == len(X_kaggle_base)
wf_test_aligned.index = X_kaggle_base.index

X_kaggle = pd.concat([X_kaggle_base, wf_test_aligned], axis=1)
X_kaggle = X_kaggle.reindex(columns=augmented_features.columns)
print(f'Kaggle test features: {X_kaggle.shape}')

proba = model.predict_proba(X_kaggle.to_numpy())[:, 1]
print(f'Predictions: mean={proba.mean():.4f}, p99={np.quantile(proba, 0.99):.4f}')

sample = pd.read_csv(ROOT / 'SepsisLabel_sample_submission.csv')
sub = pd.DataFrame({'person_id_datetime': ids, 'SepsisLabel': proba})
assert sub.shape[0] == sample.shape[0]
assert set(sub['person_id_datetime']) == set(sample['person_id_datetime'])
sub = sub.set_index('person_id_datetime').reindex(sample['person_id_datetime']).reset_index()

out_path = ARTIFACTS / 'submission_workflow.csv'
sub.to_csv(out_path, index=False)
print(f'Wrote {out_path} ({len(sub):,} rows)')
print(sub.head())

Kaggle test features: (130483, 267)


Predictions: mean=0.2052, p99=0.8800
Wrote artifacts/submission_workflow.csv (130,483 rows)
               person_id_datetime  SepsisLabel
0  1416048048_2021-03-25 10:00:00     0.436246
1   280531880_2024-01-22 18:00:00     0.241297
2  1127023302_2023-12-29 21:00:00     0.113766
3  2065909112_2021-07-07 05:00:00     0.129574
4   264445818_2024-08-23 22:00:00     0.112275
